In [1]:
# Cell 1 — Load customer messages for semantic clustering

import pandas as pd
import numpy as np

support_pairs_df = pd.read_csv(
    "../data/processed/support_pairs.csv"
)

customer_messages = (
    support_pairs_df["customer_message"]
    .fillna("")
    .astype(str)
    .str.strip()
)

customer_messages = customer_messages[
    customer_messages.str.len() > 0
].reset_index(drop=True)

print("Customer messages loaded:", len(customer_messages))
print("Unique messages:", customer_messages.str.lower().nunique())

print("\nSample messages:")
for i, message in enumerate(customer_messages.head(5), start=1):
    print(f"{i}. {message}")

Customer messages loaded: 3092
Unique messages: 3023

Sample messages:
1. What is this ? since I upgraded with this $hi&*y macOS my instant click before is now lagging to almost a minute and that disrupts my writing e-mails (work and personal) wow! this is getting crazy everyday not enjoying Apple Macbook NOW!
2. I have tried restarting my macbook but it keeps lagging. I will try rebooting and close all apps once again.
3. My iPhone been moving slow af the past couple weeks. I need answers
4. Dear god not again,
5. Noticed a bug on my @115858 iPhone X/iOS 11.1.2 While I’m on the phone I can’t close apps.


In [3]:
!pip install sentence_transformers

  Using cached sentence_transformers-6.0.1-py3-none-any.whl.metadata (20 kB)
  Using cached transformers-5.17.0-py3-none-any.whl.metadata (32 kB)
  Using cached tokenizers-0.23.2-cp310-abi3-win_amd64.whl.metadata (10 kB)
  Using cached huggingface_hub-1.31.0-py3-none-any.whl.metadata (16 kB)
  Using cached torch-2.14.0-cp314-cp314-win_amd64.whl.metadata (38 kB)
  Using cached filelock-3.32.6-py3-none-any.whl.metadata (2.0 kB)
  Using cached fsspec-2026.7.0-py3-none-any.whl.metadata (10 kB)
  Using cached hf_xet-1.6.0-cp38-abi3-win_amd64.whl.metadata (4.9 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached pyyaml-6.0.3-cp314-cp314-win_amd64.whl.metadata (2.4 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached regex-2026.9.10-cp314-cp314-win_amd64.whl.metadata (41 kB)
  Using cached typer-0.27.2-py3-none-any.whl.metadata (16 kB)
  Using cached safetensors-0.8.0-cp310-abi3-win_amd64.whl.metadata (4.2 kB)
  Using cached setuptool

In [4]:
# Cell 2 — Create semantic embeddings

from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

customer_embeddings = embedding_model.encode(
    customer_messages.tolist(),
    show_progress_bar=True,
    batch_size=32,
    normalize_embeddings=True
)

print("Embedding shape:", customer_embeddings.shape)
print("Number of messages:", len(customer_embeddings))

d:\AgentCustomerSupport\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Batches: 100%|██████████| 97/97 [00:50<00:00,  1.94it/s]

Embedding shape: (3092, 384)
Number of messages: 3092


In [5]:
# Cell 3 — Save semantic embeddings

import os
import numpy as np

os.makedirs("../data/embeddings", exist_ok=True)

embedding_path = "../data/embeddings/customer_embeddings.npy"

np.save(
    embedding_path,
    customer_embeddings
)

print("Semantic embeddings saved successfully.")
print("Shape:", customer_embeddings.shape)
print("File:", embedding_path)

Semantic embeddings saved successfully.
Shape: (3092, 384)
File: ../data/embeddings/customer_embeddings.npy


In [6]:
# Cell 4 — Evaluate candidate numbers of semantic clusters

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

candidate_k = [6, 8, 10, 12, 14, 16]
silhouette_scores = {}

for k in candidate_k:
    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )
    
    cluster_labels = kmeans.fit_predict(customer_embeddings)
    
    score = silhouette_score(
        customer_embeddings,
        cluster_labels
    )
    
    silhouette_scores[k] = score
    
    print(f"K = {k:2d} | Silhouette Score = {score:.4f}")

K =  6 | Silhouette Score = 0.0277
K =  8 | Silhouette Score = 0.0309
K = 10 | Silhouette Score = 0.0341
K = 12 | Silhouette Score = 0.0389
K = 14 | Silhouette Score = 0.0338
K = 16 | Silhouette Score = 0.0351


In [7]:
# Cell 5 — Create semantic clusters using K=12

from sklearn.cluster import KMeans

n_clusters = 12

kmeans_model = KMeans(
    n_clusters=n_clusters,
    random_state=42,
    n_init=10
)

semantic_cluster_labels = kmeans_model.fit_predict(
    customer_embeddings
)

print("Semantic clustering completed.")
print("Number of clusters:", n_clusters)

print("\nCluster sizes:")
cluster_counts = pd.Series(
    semantic_cluster_labels
).value_counts().sort_index()

print(cluster_counts)

Semantic clustering completed.
Number of clusters: 12

Cluster sizes:
0     311
1     381
2     111
3     196
4     293
5      78
6     361
7     180
8     295
9     409
10    161
11    316
Name: count, dtype: int64


In [8]:
# Cell 6 — Inspect representative messages from each semantic cluster

from sklearn.metrics import pairwise_distances
import numpy as np

# Calculate distance from every message to every cluster centroid
distances = pairwise_distances(
    customer_embeddings,
    kmeans_model.cluster_centers_,
    metric="cosine"
)

print("Representative messages by semantic cluster:\n")

for cluster_id in range(n_clusters):

    # Get indices belonging to this cluster
    cluster_indices = np.where(
        semantic_cluster_labels == cluster_id
    )[0]

    # Sort by distance to cluster centroid
    representative_indices = cluster_indices[
        np.argsort(distances[cluster_indices, cluster_id])[:8]
    ]

    print("=" * 80)
    print(f"CLUSTER {cluster_id} | Size: {len(cluster_indices)}")
    print("=" * 80)

    for i, idx in enumerate(representative_indices, start=1):
        print(f"{i}. {customer_messages.iloc[idx]}")
    
    print()

Representative messages by semantic cluster:

CLUSTER 0 | Size: 311
1. iOS 11.1
2. iOS 11.1
3. iOS 11.1
4. iOS 11.1
5. IOS 11.1
6. iOS 11.1
7. iOS 11.1
8. iOS 11.1.2

CLUSTER 1 | Size: 381
1. @AppleSupport  https://t.co/NAmOVlIDMr
2. Worked
3. @AppleSupport  https://t.co/yvu10fymce
4. @AppleSupport  https://t.co/FUUKlavB1I
5. @AppleSupport  https://t.co/3lNR4s3CWC
6. @AppleSupport  https://t.co/2vVFr3ZYiK
7. @AppleSupport  https://t.co/9KrD2oAz5L
8. HELP

CLUSTER 2 | Size: 111
1. 11.1
2. 11.1
3. 11.1
4. 11.1
5. 11.1
6. 11.1
7. 11.1
8. 11.1

CLUSTER 3 | Size: 196
1. whenever I type the letter “i” it auto corrects to a “a ?” What is going on!????
2. Yo what’s up with this glitch every time I try to type the letter “I” I
3. Okay @115858. First it was the stupid A symbol everytime I type “I”. Now it’s this for “ I.T “ .... WTF IS I.T ???????????? U ABOUT TO MAKE ME SWITCH TO @122609 WITH THESE DUMBASS GLITCHES
4. Anyone else having #IOSissues ? Everytime i type “i” it auto corrects to some

In [9]:
# Cell 7 — Identify important TF-IDF terms inside each semantic cluster

from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

cluster_keyword_vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

cluster_tfidf = cluster_keyword_vectorizer.fit_transform(
    customer_messages
)

feature_names = np.array(
    cluster_keyword_vectorizer.get_feature_names_out()
)

print("Top TF-IDF keywords by semantic cluster:\n")

for cluster_id in range(n_clusters):

    cluster_indices = np.where(
        semantic_cluster_labels == cluster_id
    )[0]

    # Mean TF-IDF score for this cluster
    cluster_scores = cluster_tfidf[
        cluster_indices
    ].mean(axis=0).A1

    top_indices = cluster_scores.argsort()[-12:][::-1]

    top_terms = feature_names[top_indices]

    print(
        f"Cluster {cluster_id:2d} "
        f"(size={len(cluster_indices):3d}): "
        + ", ".join(top_terms)
    )

Top TF-IDF keywords by semantic cluster:

Cluster  0 (size=311): ios, ios 11, 11, iphone, ios11, update, iphone ios, updated, phone, 6s, 115858, app
Cluster  1 (size=381): applesupport, yes, help, thank, thanks, https, applesupport https, hey, going, dm, email, message
Cluster  2 (size=111): 11, 10, version, version 11, updated 11, updated, thanks, 6s, 11 beta, 6s 11, just, beta
Cluster  3 (size=196): letter, type, question, 115858, fix, mark, question mark, type letter, eye, keyboard, changing, box
Cluster  4 (size=293): app, update, apps, app store, store, download, software, updated, watch, just, apple, open
Cluster  5 (size= 78): la, el, que, com, en, bateria, mi, se, 11, es, não, iphone
Cluster  6 (size=361): screen, thanks, working, just, help, macbook, issue, tried, fix, yes, problem, turn
Cluster  7 (size=180): battery, life, iphone, battery life, ios, 11, ios 11, update, phone, draining, charge, 100
Cluster  8 (size=295): 115858, phone, fix, update, https, 115858 fix, shit, wt

In [10]:
# Cell 8 — Create a semantic cluster summary for taxonomy validation

cluster_summary = []

for cluster_id in range(n_clusters):

    cluster_indices = np.where(
        semantic_cluster_labels == cluster_id
    )[0]

    # Cluster keywords
    cluster_scores = cluster_tfidf[
        cluster_indices
    ].mean(axis=0).A1

    top_indices = cluster_scores.argsort()[-10:][::-1]
    top_terms = feature_names[top_indices]

    # Representative messages
    representative_indices = cluster_indices[
        np.argsort(
            distances[cluster_indices, cluster_id]
        )[:5]
    ]

    representative_messages = [
        customer_messages.iloc[idx]
        for idx in representative_indices
    ]

    cluster_summary.append({
        "cluster_id": cluster_id,
        "size": len(cluster_indices),
        "keywords": ", ".join(top_terms),
        "representative_messages": " | ".join(
            representative_messages
        )
    })

cluster_summary_df = pd.DataFrame(cluster_summary)

print("Semantic cluster summary:")
display(cluster_summary_df)

Semantic cluster summary:


,cluster_id,size,keywords,representative_messages
0,0,311,"ios, ios 11, 11, iphone, ios11, update, iphone...",iOS 11.1 | iOS 11.1 | iOS 11.1 | iOS 11.1 | IO...
1,1,381,"applesupport, yes, help, thank, thanks, https,...",@AppleSupport https://t.co/NAmOVlIDMr | Worke...
2,2,111,"11, 10, version, version 11, updated 11, updat...",11.1 | 11.1 | 11.1 | 11.1 | 11.1
3,3,196,"letter, type, question, 115858, fix, mark, que...",whenever I type the letter “i” it auto correct...
4,4,293,"app, update, apps, app store, store, download,...",I contacted you guys and no one can help me. J...
5,5,78,"la, el, que, com, en, bateria, mi, se, 11, es",Tudo bom ??? Todos estão com o mesmo problema ...
6,6,361,"screen, thanks, working, just, help, macbook, ...",Yes I restarted it and still having the same p...
7,7,180,"battery, life, iphone, battery life, ios, 11, ...",why does my battery drain so fast now on iOS 1...
8,8,295,"115858, phone, fix, update, https, 115858 fix,...",@115858 I am so fuxking done with you and thes...
9,9,409,"iphone, phone, update, new, just, screen, ios,...",Maybe it's the phone making problems? Maybe I ...


In [12]:
# Cell 8 — Save semantic clustering results

import os
import numpy as np
import joblib

os.makedirs("../data/processed", exist_ok=True)
os.makedirs("../models", exist_ok=True)

# Save cluster labels
np.save(
    "../data/processed/semantic_cluster_labels.npy",
    semantic_cluster_labels
)

# Save trained KMeans model
joblib.dump(
    kmeans_model,
    "../models/kmeans_semantic_clusters.joblib"
)

print("Semantic clustering artifacts saved successfully.")
print(
    "Labels:",
    "../data/processed/semantic_cluster_labels.npy"
)
print(
    "KMeans model:",
    "../models/kmeans_semantic_clusters.joblib"
)
print(
    "Number of labels:",
    len(semantic_cluster_labels)
)

Semantic clustering artifacts saved successfully.
Labels: ../data/processed/semantic_cluster_labels.npy
KMeans model: ../models/kmeans_semantic_clusters.joblib
Number of labels: 3092
